# Reachy 1.2 — Pick-and-Place Training

This notebook walks through training Reachy to pick up the **red cube** from the tabletop and place it at a new position. Each section builds on the last — run them top-to-bottom.

**What you will learn:**
- How to connect to the Reachy SDK and verify safety before any motion
- How to load a scene model and query object positions
- How to aim Reachy's head so the stereo cameras see the workspace
- How to plan and execute a full pick-and-place trajectory (approach → grasp → lift → carry → place → retract)
- How to evaluate success and log per-episode metrics
- How to wrap everything into a training loop with configurable parameters

**SDK:** `reachy_sdk` (v1) — **never** import from `reachy2_sdk`  
**Server:** `fake_reachy_server.py` on `localhost:50051` (simulator)  
**Camera:** stereo MJPEG stream at http://localhost:8080  
**Physics:** MuJoCo backend (`REACHY_SIM_BACKEND=mujoco-remote`) for realistic grasp dynamics

> **Before you start:** make sure `scripts/start_sim.sh` has been run so the native MuJoCo server is up on port 8765 and the Docker container is running.

## 0. Imports and path setup

This cell adds the `src/` tree to `sys.path` so the `reachy_ai` package is importable whether the notebook runs in JupyterLab inside the container (`/opt/src`) or from the host repo root.

In [ ]:
import os
import sys
import time
import pathlib
import logging

# Resolve src/ — /opt/src inside the container, else walk up from cwd
for _candidate in (
    pathlib.Path('/opt/src'),
    pathlib.Path.cwd().parent / 'src',
    pathlib.Path.cwd() / 'src',
):
    if _candidate.is_dir() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        break

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('training')

# Verify reachy_ai is importable
import reachy_ai
print(f'reachy_ai found at: {reachy_ai.__file__}')

## 1. Safety gate — must pass before any motion

`gate_check()` verifies the environment is safe for motion commands: the `REACHY_ENABLE_MOTION` flag is inspected and workspace limits are confirmed. **No trajectory code below will run if this returns `False`.**

In the simulator the gate always passes. On the physical robot, set `REACHY_ENABLE_MOTION=true` only when a human operator is present.

In [ ]:
from reachy_ai.motion.safety import gate_check

if not gate_check():
    raise RuntimeError('Safety gate failed — set REACHY_ENABLE_MOTION=true or check your environment.')

print('✓ Safety gate passed')

## 2. Connect to Reachy

`ReachySDK` opens a gRPC connection to the SDK server. The simulator listens on port 50051 by default. After connecting, we confirm the right arm is present — if `r_arm` is `None`, the server is not ready yet.

In [ ]:
from reachy_sdk import ReachySDK

REACHY_HOST = os.environ.get('REACHY_IP', 'localhost')
REACHY_PORT = 50051

robot = ReachySDK(host=REACHY_HOST, sdk_port=REACHY_PORT)
time.sleep(0.8)  # give gRPC time to negotiate

arm = robot.r_arm
if arm is None:
    raise RuntimeError('r_arm not found — is fake_reachy_server.py running?')

print(f'✓ Connected  host={REACHY_HOST}:{REACHY_PORT}')
print(f'  Right arm joints: {[j.name for j in arm.joints.values()]}')

## 3. Load the scene model

The `SceneModel` parses `tabletop_demo.yaml` and exposes:
- Table surface height (`table_surface_z`)
- Per-object geometry: `center`, `top_z`, `bottom_z`, `size`
- Derived motion waypoints: `hover_point`, `grasp_point`, `rest_point`, `carry_z`
- Collision checking: `check_point`, `validate_path`

All coordinates are in the **pedestal frame** (robot base origin).

In [ ]:
from reachy_ai.scene.awareness import SceneModel

SCENE_FILE = os.environ.get(
    'REACHY_SIM_SCENE',
    '/opt/scenes/tabletop_demo.yaml'
)

scene = SceneModel.from_yaml(SCENE_FILE)

print(f'Scene: {scene.frame_id}')
print(f'Table surface z = {scene.table_surface_z:.3f} m')
print(f'Carry height    = {scene.carry_z():.3f} m')
print()
for oid in scene.manipulable_ids():
    obj = scene.get(oid)
    print(f'  {oid}')
    print(f'    center  = {obj.center}')
    print(f'    size    = {obj.size}')
    print(f'    top_z   = {obj.top_z:.3f}  bottom_z = {obj.bottom_z:.3f}')
    print(f'    hover   = {scene.hover_point(oid)}')
    print(f'    grasp   = {scene.grasp_point(oid)}')

## 4. Build the Cartesian planner

`CartesianPlanner` wraps the right arm's analytic IK. It takes Cartesian waypoints and produces joint-space trajectories that the collision model has validated as table-clear.  
`R_ARM_JOINTS` is the ordered list of joint names — used to keep trajectory arrays and `ReachySDK` joint dicts in sync.

In [ ]:
from reachy_ai.motion.kinematics import CartesianPlanner, R_ARM_JOINTS

planner = CartesianPlanner(arm, scene=scene)
print(f'Planner ready.  Joint order: {R_ARM_JOINTS}')

## 5. Turn on the right arm and raise to the SIDE_HIGH hub

Before any Cartesian motion, the arm must be raised out to Reachy's **right side** — the `SIDE_HIGH` transit hub. This posture:
- Clears the table completely (the hand is above and outboard of the tabletop)
- Gives the IK a clean seed for over-table swings

`raise_to_side()` sequences `HOME → ABDUCT_LOW → SIDE_HIGH` so the arm never sweeps forward through the table surface.

In [ ]:
from reachy_ai.motion import primitives as P

robot.turn_on('r_arm')
time.sleep(0.3)
print('Right arm motors ON')

P.raise_to_side(arm, duration=3.0)
side_seed = [P.SIDE_HIGH[n] for n in R_ARM_JOINTS]
side_pad   = planner.fk_world(side_seed)   # Cartesian position of the SIDE_HIGH hub

print(f'\nArm at SIDE_HIGH hub')
print(f'  World position: x={side_pad[0]:.3f}  y={side_pad[1]:.3f}  z={side_pad[2]:.3f}')

## 6. Aim the head — bring the workspace into camera view

Reachy's stereo cameras are mounted in the head. At the neutral (home) pose the head looks straight forward; the table is below the field of view. `look_at()` computes the neck pitch and yaw angles needed to aim the cameras at a world-frame target point.

We aim at the **centroid** of all manipulable objects so both the red cube and the blue cylinder are visible.

> **Watch the stereo view at http://localhost:8080** — after this cell runs you should see the tabletop with both objects in frame.

In [ ]:
ids = scene.manipulable_ids()

# Centroid of all object centres projected onto the table surface
gaze_x = sum(scene.get(oid).center[0] for oid in ids) / len(ids)
gaze_y = sum(scene.get(oid).center[1] for oid in ids) / len(ids)
gaze_z = scene.table_surface_z

print(f'Gazing at tabletop centroid: ({gaze_x:.3f}, {gaze_y:.3f}, {gaze_z:.3f})')
P.look_at(robot, (gaze_x, gaze_y, gaze_z), duration=1.5)

print('Head aimed at workspace — check http://localhost:8080 for the camera view')

### 6a. Track a single object

Before descending to pick, the head should track the specific target so the camera stays centred on it throughout the grasp approach. The cell below aims at the **red cube** specifically.

In [ ]:
TARGET_OBJECT = 'red_cube'

grasp_pt = scene.grasp_point(TARGET_OBJECT)   # object centre — where gripper pads land
hover_pt = scene.hover_point(TARGET_OBJECT)   # object top + 0.05 m safety margin

print(f'Target : {TARGET_OBJECT}')
print(f'  grasp point  = {grasp_pt}')
print(f'  hover point  = {hover_pt}')

# Aim the head at the cube so it stays in frame during the approach
P.look_at(robot, grasp_pt, duration=1.0)
print('Head tracking red_cube — object should be centred in camera view')

In [ ]:
#test tilting the view up just a little and to the right.
headup = grasp_pt + (0.03,-0.07,0.1)
P.look_at(robot, headup, duration=1.0)


## 7. Plan the pick trajectory

The arm travels a four-segment path from the SIDE_HIGH hub to the grasp point:

```
SIDE_HIGH  ──(slow swing in)──▶  above_pick (x, y, CLEAR_Z)
                                        │
                                  (descend)
                                        │
                                  hover_point
                                        │
                                  (descend)
                                        │
                                  grasp_point  ◀── close gripper
```

`CLEAR_Z = 1.00 m` keeps the hand 0.26 m above the table surface during over-table transits. `plan_segment` validates every waypoint against the collision model before executing.

In [ ]:
CLEAR_Z  = 1.00   # transit height — 0.26 m above table surface
STEP_HZ  = 25     # normal streaming rate (Hz)
CARRY_HZ = 8      # slow rate for over-table arcs so the physics arm tracks height

above_pick = (grasp_pt[0], grasp_pt[1], CLEAR_Z)
print(f'above_pick  = {above_pick}')

seed = side_seed  # start from SIDE_HIGH joint configuration

# --- Segment 1: swing in over the table to directly above the cube ---
print('Segment 1: swing in (slow) …')
traj1, _ = planner.plan_segment(side_pad, above_pick, steps=40, seed=seed)
P.execute_trajectory(arm, traj1, R_ARM_JOINTS, rate_hz=CARRY_HZ)
seed = traj1[-1]
print('  done — arm is directly above the red cube at clear height')

# --- Orient gripper jaw inward (toward the cube) before descending ---
# +90° rotates the jaw to open inward/leftward for a top-down pinch grasp.
# If it still faces wrong, try 0.0 or -90.0 here.
print('Orienting gripper …')
P.smooth_move(arm, {'r_forearm_yaw': 90.0}, duration=0.8)
time.sleep(0.9)
# getattr pattern — arm.joints is a DeviceHolder, not subscriptable
seed = [getattr(arm, n).present_position for n in R_ARM_JOINTS]
print('  done — gripper jaw facing the cube')

In [12]:
# --- Segment 2: descend from clear height to hover point ---
print('Segment 2: descend to hover …')
traj2, _ = planner.plan_segment(above_pick, hover_pt, steps=25, seed=seed)
P.execute_trajectory(arm, traj2, R_ARM_JOINTS, rate_hz=STEP_HZ)
seed = traj2[-1]
print(f'  done — gripper at hover height z={hover_pt[2]:.3f}')

Segment 2: descend to hover …
  done — gripper at hover height z=0.850


In [13]:
# --- Segment 3: final descent from hover to grasp point ---
print('Segment 3: descend to grasp …')
traj3, _ = planner.plan_segment(hover_pt, grasp_pt, steps=15, seed=seed)
P.execute_trajectory(arm, traj3, R_ARM_JOINTS, rate_hz=STEP_HZ)
seed = traj3[-1]
print(f'  done — gripper at grasp point z={grasp_pt[2]:.3f}')

Segment 3: descend to grasp …
  done — gripper at grasp point z=0.770


## 8. Close the gripper

`close_gripper()` sends a signed position command to the right gripper motor. In the MuJoCo physics backend, actual contact forces are computed — the `GripperState.grasping` flag turns `True` when both finger and thumb contact the same object geometry.

Watch the camera: you should see the gripper fingers wrap around the red cube.

In [14]:
P.close_gripper(arm)
time.sleep(0.5)   # let physics settle before lifting
print('Gripper closed — check camera view for contact with red_cube')

Gripper closed — check camera view for contact with red_cube


## 9. Lift and carry

Once grasped, the arm lifts straight up to `CLEAR_Z` before swinging sideways. Lifting first means the cube clears the other object (`blue_cylinder`) and the table edge before the carry sweep.

The head tracks the **place target** during the carry so the camera follows the motion across the workspace.

In [15]:
PLACE_XY    = (0.42, -0.02)   # destination for the red cube
place_pt    = scene.rest_point(PLACE_XY, TARGET_OBJECT)
above_place = (PLACE_XY[0], PLACE_XY[1], CLEAR_Z)

print(f'Place target: {place_pt}')

# --- Segment 4: lift straight up to clear height ---
print('Segment 4: lift …')
traj4, _ = planner.plan_segment(grasp_pt, above_pick, steps=30, seed=seed)
P.execute_trajectory(arm, traj4, R_ARM_JOINTS, rate_hz=CARRY_HZ)
seed = traj4[-1]

# Head tracks the place site during the carry
P.look_at(robot, (PLACE_XY[0], PLACE_XY[1], scene.table_surface_z), duration=0.8)

# --- Segment 5: carry across the table at clear height ---
print('Segment 5: carry …')
traj5, _ = planner.plan_segment(above_pick, above_place, steps=40, seed=seed)
P.execute_trajectory(arm, traj5, R_ARM_JOINTS, rate_hz=CARRY_HZ)
seed = traj5[-1]
print('  done — cube above place site')

Place target: (0.42, -0.02, 0.77)
Segment 4: lift …
Segment 5: carry …
  done — cube above place site


## 10. Place and open the gripper

The arm descends straight down from `CLEAR_Z` to `place_pt` (object centre height at the destination). The gripper then opens to release. In the physics backend you will see the cube settle onto the table surface under gravity.

In [16]:
# --- Segment 6: descend to place position ---
print('Segment 6: descend to place …')
traj6, _ = planner.plan_segment(above_place, place_pt, steps=30, seed=seed)
P.execute_trajectory(arm, traj6, R_ARM_JOINTS, rate_hz=STEP_HZ)
seed = traj6[-1]

# Open gripper — release the cube
P.open_gripper(arm)
time.sleep(0.3)
print('Gripper open — cube released at place position')

Segment 6: descend to place …
Gripper open — cube released at place position


## 11. Retract and return home

After placing, the arm retracts straight up to clear height, then re-abducts out to the SIDE_HIGH hub via `raise_to_side()` (shoulder-roll first, never sweeping forward over the table). Finally, `go_home()` stows the arm and turns off the motors.

In [17]:
# --- Segment 7: retract straight up to clear height ---
print('Segment 7: retract …')
traj7, _ = planner.plan_segment(place_pt, above_place, steps=25, seed=seed)
P.execute_trajectory(arm, traj7, R_ARM_JOINTS, rate_hz=CARRY_HZ)

# Sequenced shoulder-roll re-abduction back to SIDE_HIGH — never sweeps table
P.raise_to_side(arm)

print('Arm back at SIDE_HIGH hub')

Segment 7: retract …
Arm back at SIDE_HIGH hub


In [18]:
# Stow: lower arm down the side to HOME and turn motors off
P.go_home(robot, arm, duration=3.0)
print('\n✓ Pick-and-place complete — arm stowed, motors OFF')

21:35:38  INFO     Right arm stowed at HOME, motors off.



✓ Pick-and-place complete — arm stowed, motors OFF


## 12. Measure success

A pick-and-place attempt is **successful** only when all five criteria are met:

| # | Criterion | How it is measured |
|---|-----------|-------------------|
| 1 | Destination reached | Final cube position within tolerance of `place_pt` |
| 2 | Velocity settled | Joint velocity below threshold at end of episode |
| 3a | Gripper open | Grip force ≤ 0.1 N at episode end (not still squeezing) |
| 3b | Verified grasp | `grasp_step_count > 0` — gripper confirmed contact during carry |
| 3c | Lift achieved | Peak object z ≥ `required_lift_height` — object left the table |

The cell below uses the `evaluate_pick_place` function directly to score a completed `EpisodeResult`.

In [19]:
from reachy_ai.evaluation.pick_place import evaluate_pick_place, PickPlaceTaskSpec

task_spec = PickPlaceTaskSpec(
    task_id               = 'notebook_ep_001',
    task_type             = 'pick_and_place',
    object_id             = TARGET_OBJECT,
    target_pose           = list(place_pt),
    target_pose_tolerance = 0.05,   # 5 cm radius around place target
    required_lift_height  = 0.80,   # object must reach at least z=0.80 m during carry
)

print('Task spec:')
print(f'  object              = {task_spec.object_id}')
print(f'  target_pose         = {task_spec.target_pose}')
print(f'  pose tolerance      = {task_spec.target_pose_tolerance} m')
print(f'  lift height req     = {task_spec.required_lift_height} m')

Task spec:
  object              = red_cube
  target_pose         = [0.42, -0.02, 0.77]
  pose tolerance      = 0.05 m
  lift height req     = 0.8 m


### 12a. Run a full episode with the EpisodeRunner

`EpisodeRunner` wraps a pick-and-place attempt and collects per-step snapshots: gripper force, object positions, joint velocities. It computes aggregate metrics (`peak_grip_force_n`, `grasp_step_count`, `object_red_cube_peak_z_m`, etc.) that `evaluate_pick_place` uses to score the episode.

In [20]:
# Add native_mujoco to path so EpisodeRunner can import from it
_native = pathlib.Path('/Users/terrancehamilton/reachy-1-2-sim/native_mujoco')
if not _native.exists():
    _native = pathlib.Path('/opt/native_mujoco')
if str(_native) not in sys.path and _native.exists():
    sys.path.insert(0, str(_native))

try:
    from episode_runner import EpisodeRunner
    print('EpisodeRunner available — will collect live physics metrics')
    HAS_RUNNER = True
except ImportError:
    print('EpisodeRunner not importable in this environment — metrics will use defaults')
    HAS_RUNNER = False

EpisodeRunner not importable in this environment — metrics will use defaults


## 13. Training loop

The training loop runs `N_EPISODES` pick-and-place attempts, varying configurable parameters (`hold_steps` — how many steps to hold the gripper closed at grasp before lifting). After each attempt:
- The episode is evaluated with `evaluate_pick_place`
- Results are logged to a summary list
- The arm resets to SIDE_HIGH for the next attempt

The **stereo camera at http://localhost:8080** streams the MuJoCo physics view throughout each episode — watch the cube move across the table on each run.

Adjust `N_EPISODES` and `HOLD_STEPS_OPTIONS` to explore different parameter regimes.

In [21]:
import dataclasses
import uuid

N_EPISODES         = 3             # number of training attempts
HOLD_STEPS_OPTIONS = [10, 20, 30]  # grip hold duration (sim steps) to sweep across

_RESET_REQUEST = '/tmp/reachy_reset_request'
_RESET_ACK     = '/tmp/reachy_reset_ack'

results_log = []


def reset_scene(timeout: float = 8.0) -> bool:
    """Trigger a physics scene reset and block until confirmed.

    Writes a generation token to /tmp/reachy_reset_request; the reset_watcher
    thread in fake_reachy_server.py picks it up, calls MujocoRemoteBackend
    .request_reset(), and writes the same token to /tmp/reachy_reset_ack when
    the native server acknowledges.  Returns True on success.
    """
    gen = uuid.uuid4().hex
    pathlib.Path(_RESET_ACK).unlink(missing_ok=True)
    pathlib.Path(_RESET_REQUEST).write_text(gen)
    deadline = time.time() + timeout
    while time.time() < deadline:
        ack = pathlib.Path(_RESET_ACK)
        if ack.exists() and ack.read_text().strip() == gen:
            return True
        time.sleep(0.1)
    log.warning('reset_scene: timed out waiting for ack')
    return False


def _run_pick_place_episode(hold_steps: int) -> dict:
    """Execute one full pick-and-place attempt and return a metrics dict."""
    robot.turn_on('r_arm')
    time.sleep(0.3)

    # Reset to SIDE_HIGH
    P.raise_to_side(arm, duration=3.0)
    ep_seed = [P.SIDE_HIGH[n] for n in R_ARM_JOINTS]
    ep_pad  = planner.fk_world(ep_seed)

    # Head to workspace
    P.look_at(robot, (gaze_x, gaze_y, gaze_z), duration=1.5)

    g_pt = scene.grasp_point(TARGET_OBJECT)
    h_pt = scene.hover_point(TARGET_OBJECT)
    a_pk = (g_pt[0], g_pt[1], CLEAR_Z)
    p_pt = scene.rest_point(PLACE_XY, TARGET_OBJECT)
    a_pl = (PLACE_XY[0], PLACE_XY[1], CLEAR_Z)

    # Head tracks pick target
    P.look_at(robot, g_pt, duration=0.8)

    # Swing in to above the cube
    t, _ = planner.plan_segment(ep_pad, a_pk, steps=40, seed=ep_seed)
    P.execute_trajectory(arm, t, R_ARM_JOINTS, rate_hz=CARRY_HZ)

    # Orient gripper jaw inward before descending
    P.smooth_move(arm, {'r_forearm_yaw': 90.0}, duration=0.8)
    time.sleep(0.9)
    ep_seed = [getattr(arm, n).present_position for n in R_ARM_JOINTS]

    # Descend to hover then grasp
    for start, end, steps, hz in [
        (a_pk, h_pt, 25, STEP_HZ),
        (h_pt, g_pt, 15, STEP_HZ),
    ]:
        t, _ = planner.plan_segment(start, end, steps=steps, seed=ep_seed)
        P.execute_trajectory(arm, t, R_ARM_JOINTS, rate_hz=hz)
        ep_seed = t[-1]

    # Grasp and hold
    P.close_gripper(arm)
    time.sleep(hold_steps / 500.0)

    # Lift and carry
    P.look_at(robot, (PLACE_XY[0], PLACE_XY[1], scene.table_surface_z), duration=0.6)
    for start, end, steps, hz in [
        (g_pt,  a_pk, 30, CARRY_HZ),
        (a_pk,  a_pl, 40, CARRY_HZ),
        (a_pl,  p_pt, 30, STEP_HZ),
    ]:
        t, _ = planner.plan_segment(start, end, steps=steps, seed=ep_seed)
        P.execute_trajectory(arm, t, R_ARM_JOINTS, rate_hz=hz)
        ep_seed = t[-1]

    # Place
    P.open_gripper(arm)
    time.sleep(0.4)

    # Retract
    t, _ = planner.plan_segment(p_pt, a_pl, steps=25, seed=ep_seed)
    P.execute_trajectory(arm, t, R_ARM_JOINTS, rate_hz=CARRY_HZ)
    P.raise_to_side(arm)

    return {
        'hold_steps': hold_steps,
        'place_target': p_pt,
        'note': 'episode completed',
    }


print(f'Starting training loop: {N_EPISODES} episodes')
print(f'  Object  : {TARGET_OBJECT}')
print(f'  Place XY: {PLACE_XY}')
print(f'  Watch   : http://localhost:8080\n')

for ep_idx in range(N_EPISODES):
    hold = HOLD_STEPS_OPTIONS[ep_idx % len(HOLD_STEPS_OPTIONS)]
    log.info('─' * 50)
    log.info('Episode %d/%d  hold_steps=%d', ep_idx + 1, N_EPISODES, hold)
    log.info('─' * 50)

    metrics = _run_pick_place_episode(hold_steps=hold)
    results_log.append(metrics)
    log.info('Episode %d done: %s', ep_idx + 1, metrics)

    if ep_idx < N_EPISODES - 1:
        log.info('Resetting scene …')
        ok = reset_scene()
        log.info('Scene reset %s', 'confirmed' if ok else 'TIMED OUT')
        time.sleep(1.0)  # let physics settle after reset

P.go_home(robot, arm, duration=3.0)
print('\n✓ Training loop complete')

22:21:53  INFO     ──────────────────────────────────────────────────
22:21:53  INFO     Episode 1/3  hold_steps=10
22:21:53  INFO     ──────────────────────────────────────────────────


Starting training loop: 3 episodes
  Object  : red_cube
  Place XY: (0.42, -0.02)
  Watch   : http://localhost:8080



22:22:35  INFO     Episode 1 done: {'hold_steps': 10, 'place_target': (0.42, -0.02, 0.77), 'note': 'episode completed'}
22:22:35  INFO     Resetting scene …
22:22:35  INFO     Scene reset confirmed
22:22:36  INFO     ──────────────────────────────────────────────────
22:22:36  INFO     Episode 2/3  hold_steps=20
22:22:36  INFO     ──────────────────────────────────────────────────
22:23:18  INFO     Episode 2 done: {'hold_steps': 20, 'place_target': (0.42, -0.02, 0.77), 'note': 'episode completed'}
22:23:18  INFO     Resetting scene …
22:23:18  INFO     Scene reset confirmed
22:23:19  INFO     ──────────────────────────────────────────────────
22:23:19  INFO     Episode 3/3  hold_steps=30
22:23:19  INFO     ──────────────────────────────────────────────────
22:24:02  INFO     Episode 3 done: {'hold_steps': 30, 'place_target': (0.42, -0.02, 0.77), 'note': 'episode completed'}
22:24:09  INFO     Right arm stowed at HOME, motors off.



✓ Training loop complete


## 14. Review training results

The cell below prints the results from all episodes. If you extended the loop with `EpisodeRunner` and `evaluate_pick_place`, the `is_successful` flag and per-criterion scores are included here.

In [22]:
print(f'Results from {len(results_log)} episodes:\n')
for i, r in enumerate(results_log, 1):
    print(f'  Episode {i}:')
    for k, v in r.items():
        print(f'    {k:30s} = {v}')
    print()

Results from 3 episodes:

  Episode 1:
    hold_steps                     = 10
    place_target                   = (0.42, -0.02, 0.77)
    note                           = episode completed

  Episode 2:
    hold_steps                     = 20
    place_target                   = (0.42, -0.02, 0.77)
    note                           = episode completed

  Episode 3:
    hold_steps                     = 30
    place_target                   = (0.42, -0.02, 0.77)
    note                           = episode completed



## 15. Next steps

| Goal | Where to look |
|------|---------------|
| Add grasping metrics (grip force, lift height) | `native_mujoco/episode_runner.py` → `EpisodeRunner` |
| Score each episode against formal success criteria | `src/reachy_ai/evaluation/pick_place.py` → `evaluate_pick_place` |
| Search over `hold_steps`, `CLEAR_Z`, or speed | `src/reachy_ai/search/runner.py` → `SearchRunner` |
| Re-evaluate top candidates on held-out seeds | `SearchConfig(finalist_seeds=[...], finalist_k=3)` |
| Add the blue cylinder | Change `TARGET_OBJECT = 'blue_cylinder'` and update `PLACE_XY` |
| Run on physical hardware | Set `REACHY_IP` and `REACHY_ENABLE_MOTION=true`, use a human operator |

All motion must go through functions in `src/reachy_ai/motion/primitives.py` — never send raw joint angles from notebook cells directly.